In [1]:
!pip install transformers torch

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [4]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Decode response
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.



You:  Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hi! :D


You:  What is AI?


Chatbot: Artificial Intelligence!


You:  Who created Python?


Chatbot: No, the AI created the AI.


You:  exit


Chatbot: Goodbye! 👋


In [5]:
max_length=200

In [6]:
response = response.strip()

In [7]:
if response == "":
    response = "I'm not sure how to respond to that."